In [2]:
import sqlite3
print(sqlite3.sqlite_version)

3.50.4


In [ ]:
import sqlite3

conn = sqlite3.connect("empresa.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id INTEGER PRIMARY KEY,
    nome TEXT
)
""")

conn.commit()
conn.close()

print("Base de dados criada!")

In [2]:
!pip install ortools


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/23.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.9 MB ? eta -:--:--
   --- ------------------------------------ 2.1/23.9 MB 6.5 MB/s eta 0:00:04
   --------- ------------------------------ 5.8/23.9 MB 11.3 MB/s eta 0:00:02
   ------------------ --------------------- 11.0/23.9 MB 15.4 MB/s eta 0:00:01
   --------------------------- ------------ 16.3/23.9 MB 17.7 MB/s eta 0:00:01
   ---------------------------------- ----- 20.4/23.9 MB 18.0 MB/s eta 0:00:01
   ---------------------------------------  23.9/23.9 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------- 23.9/23.9 MB 17.2 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   --------------- ------------------------ 4.7/12.4 MB 23.8 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.4 MB 23.0 MB/s eta 0:00:01
   ------

In [3]:
import ortools
print(ortools.__version__)

9.15.6755


In [7]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model():
    data = {}
    data['distance_matrix'] = [
        [0, 15, 20, 12, 25, 18, 22],
        [15, 0, 10, 14, 21, 13, 17],
        [20, 10, 0, 19, 11, 22, 14],
        [12, 14, 19, 0, 23, 15, 28],
        [25, 21, 11, 23, 0, 16, 12],
        [18, 13, 22, 15, 16, 0, 9],
        [22, 17, 14, 28, 12, 9, 0],
    ]
    data['time_matrix'] = data['distance_matrix'] 
    
    data['demands'] = [0, 2, 2, 3, 1, 2, 1]        
    data['vehicle_capacities'] = [5, 5, 5]         
    data['num_vehicles'] = 3
    data['depot'] = 0

    data['time_windows'] = [
        (0, 300),   
        (10, 100),  
        (20, 120),  
        (10, 150),  
        (50, 200),  
        (30, 180),  
        (40, 220),  
    ]
    data['service_times'] = [0, 15, 10, 20, 15, 10, 10]
    return data

def main():
    data = create_data_model()
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    # Distância
    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Tempo
    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['time_matrix'][from_node][to_node] + data['service_times'][from_node]
    time_callback_index = routing.RegisterTransitCallback(time_callback)

    # Carga
    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]
    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    # Dimensões (As tuas Restrições)
    routing.AddDimensionWithVehicleCapacity(demand_callback_index, 0, data['vehicle_capacities'], True, 'Capacity')
    
    routing.AddDimension(time_callback_index, 60, 300, True, 'Time')
    time_dimension = routing.GetDimensionOrDie('Time')
    for location_node, time_window in enumerate(data['time_windows']):
        if location_node == 0: continue
        index = manager.NodeToIndex(location_node)
        time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])

    routing.AddDimension(transit_callback_index, 0, 65, True, 'Distance')
    
    routing.AddDimension(routing.RegisterTransitCallback(lambda f, t: 1), 0, 4, True, 'Stops')

    target_customer_index = manager.NodeToIndex(3)
    routing.VehicleVar(target_customer_index).SetValue(2)

    # Execução
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print_solution(data, manager, routing, solution)
    else:
        print("Inviável: O motor testou todas as hipóteses e as tuas restrições são demasiado severas para encontrar uma rota legal.")

def print_solution(data, manager, routing, solution):
    time_dimension = routing.GetDimensionOrDie('Time')
    distance_dimension = routing.GetDimensionOrDie('Distance')
    capacity_dimension = routing.GetDimensionOrDie('Capacity')
    
    print("=== RELATÓRIO DETALHADO DE OPERAÇÃO ===\n")
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        
        end_index = routing.End(vehicle_id)
        if solution.Value(distance_dimension.CumulVar(end_index)) == 0:
            print(f'🚛 Camião {vehicle_id}: Sem serviço (Ficou no Depósito).\n')
            continue
            
        plan_output = f'🚛 ROTA DO CAMIÃO {vehicle_id}:\n'
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            
            time_var = time_dimension.CumulVar(index)
            dist_var = distance_dimension.CumulVar(index)
            load_var = capacity_dimension.CumulVar(index)
            
            chegada_min = solution.Min(time_var)
            partida_min = chegada_min + data['service_times'][node_index]
            carga = solution.Value(load_var)
            distancia = solution.Value(dist_var)
            
            hora_chegada = f"{8 + chegada_min // 60:02d}:{chegada_min % 60:02d}"
            hora_partida = f"{8 + partida_min // 60:02d}:{partida_min % 60:02d}"
            
            if node_index == 0:
                plan_output += f"  📍 INÍCIO | Depósito Central\n"
                plan_output += f"      ⏰ Saída: {hora_partida}\n"
                plan_output += f"      🛣️ Odómetro: 0 km\n"
            else:
                volume = data['demands'][node_index]
                plan_output += f"  📍 PARAGEM | Cliente {node_index}\n"
                plan_output += f"      ⏰ Chegada: {hora_chegada} | ⏳ Partida: {hora_partida} (Serviço: {data['service_times'][node_index]} min)\n"
                plan_output += f"      📦 Descarga: {volume} un. | 🚛 Carga na rota: {carga}/{data['vehicle_capacities'][vehicle_id]}\n"
                plan_output += f"      🛣️ Odómetro parcial: {distancia} km\n"
                
            plan_output += "           ⬇️\n"
            index = solution.Value(routing.NextVar(index))
        
        node_index = manager.IndexToNode(index)
        time_var = time_dimension.CumulVar(index)
        dist_var = distance_dimension.CumulVar(index)
        
        chegada_min = solution.Min(time_var)
        distancia_total = solution.Value(dist_var)
        hora_chegada = f"{8 + chegada_min // 60:02d}:{chegada_min % 60:02d}"
        
        plan_output += f"  🏁 FIM DE TURNO | Regresso ao Depósito\n"
        plan_output += f"      ⏰ Hora de Chegada: {hora_chegada}\n"
        plan_output += f"      🛣️ Distância Total Percorrida: {distancia_total} km\n\n"
        
        print(plan_output)

# Esta linha é o que dá o "tiro de partida" ao código no Python
if __name__ == '__main__':
    main()

=== RELATÓRIO DETALHADO DE OPERAÇÃO ===

🚛 ROTA DO CAMIÃO 0:
  📍 INÍCIO | Depósito Central
      ⏰ Saída: 08:00
      🛣️ Odómetro: 0 km
           ⬇️
  📍 PARAGEM | Cliente 1
      ⏰ Chegada: 08:15 | ⏳ Partida: 08:30 (Serviço: 15 min)
      📦 Descarga: 2 un. | 🚛 Carga na rota: 0/5
      🛣️ Odómetro parcial: 15 km
           ⬇️
  📍 PARAGEM | Cliente 2
      ⏰ Chegada: 08:40 | ⏳ Partida: 08:50 (Serviço: 10 min)
      📦 Descarga: 2 un. | 🚛 Carga na rota: 2/5
      🛣️ Odómetro parcial: 25 km
           ⬇️
  🏁 FIM DE TURNO | Regresso ao Depósito
      ⏰ Hora de Chegada: 09:10
      🛣️ Distância Total Percorrida: 45 km


🚛 ROTA DO CAMIÃO 1:
  📍 INÍCIO | Depósito Central
      ⏰ Saída: 08:00
      🛣️ Odómetro: 0 km
           ⬇️
  📍 PARAGEM | Cliente 5
      ⏰ Chegada: 08:30 | ⏳ Partida: 08:40 (Serviço: 10 min)
      📦 Descarga: 2 un. | 🚛 Carga na rota: 0/5
      🛣️ Odómetro parcial: 18 km
           ⬇️
  📍 PARAGEM | Cliente 6
      ⏰ Chegada: 08:49 | ⏳ Partida: 08:59 (Serviço: 10 min)
      📦